## Cell 1: Install dependencies

In [ ]:
# ── Cell 1: Install dependencies (numpy 1.26.4 compatible) ───────────────────
import subprocess, sys

def pip(*args, no_deps=True):
    cmd = [sys.executable, "-m", "pip", "install", "-q"]
    if no_deps:
        cmd.append("--no-deps")
    cmd.extend(args)
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"❌ Failed: {args}\n{result.stderr[-300:]}")
    else:
        print(f"✅ {args[0]}")

pip("timm==0.6.13")
pip("albumentations==1.3.1")
pip("grad-cam==1.4.6")
pip("imbalanced-learn==0.12.3", no_deps=False)  # ← with deps, fixes parse_version
pip("transformers==4.38.2")
pip("accelerate==0.27.2")
pip("qudida==0.0.4")

print()
print("=" * 55)
print("  ⚠️  RESTART THE KERNEL NOW before continuing!")
print("  Kaggle:  Run → Restart & Clear Output")
print("  Then run from Cell 2 onwards.")
print("  Do NOT re-run Cell 1 after restarting.")
print("=" * 55)

## Cell 2: Imports

In [ ]:
import os, gc, json, warnings, random, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

import torchvision.transforms as T
import timm

import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, cohen_kappa_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)
from sklearn.preprocessing import label_binarize
from imblearn.over_sampling import SMOTE

from scipy import stats

warnings.filterwarnings('ignore')

# ─── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)

seed_everything()

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

In [ ]:
import os

for root, dirs, files in os.walk('/kaggle/input/datasets'):
    level = root.replace('/kaggle/input/datasets', '').count(os.sep)
    if level < 3:
        indent = ' ' * 2 * level
        print(f'{indent}{os.path.basename(root)}/')

## Cell 3: Configuration

In [ ]:
class CFG:
    # ── Paths ────────────────────────────────────────────────────────────
    APTOS_DIR = '/kaggle/input/datasets/sovitrath/diabetic-retinopathy-224x224-gaussian-filtered'
    IDRID_DIR  = '/kaggle/input/datasets/aaryapatel98/indian-diabetic-retinopathy-image-dataset'
    OUT_DIR     = '/kaggle/working/outputs'

    # ── Image ─────────────────────────────────────────────────────────────
    IMG_SIZE    = 224
    MEAN        = [0.485, 0.456, 0.406]
    STD         = [0.229, 0.224, 0.225]

    # ── Training ──────────────────────────────────────────────────────────
    N_FOLDS     = 5
    EPOCHS   = 30   # reduced from 50
    BATCH_SIZE  = 32
    NUM_WORKERS = 0
    N_CLASSES   = 5

    # ── Optimiser ─────────────────────────────────────────────────────────
    LR_HEAD     = 1e-4
    LR_BACKBONE = 1e-5
    WEIGHT_DECAY= 1e-4
    T0          = 10

    # ── Early stopping ────────────────────────────────────────────────────
    PATIENCE = 7    # reduced from 10

    # ── XAI ──────────────────────────────────────────────────────────────
    SALIENCY_THRESH_PERCENTILE = 70

    # ── Ablation model keys ───────────────────────────────────────────────
    MODELS = ['efficientnet_only', 'vit_only', 'hybrid']

os.makedirs(CFG.OUT_DIR, exist_ok=True)

# ── Verify dataset paths are mounted correctly ────────────────────────────
import os
for name, path in [('APTOS', CFG.APTOS_DIR), ('IDRiD', CFG.IDRID_DIR)]:
    if os.path.exists(path):
        print(f'{name} found at {path}')
        print(f'  Contents: {os.listdir(path)}')
    else:
        print(f'WARNING: {name} NOT found at {path}')
        print('  Check that the dataset is added to this notebook in the right panel.')


In [ ]:
import os

# Find actual image location
for root, dirs, files in os.walk('/kaggle/input/datasets/sovitrath/diabetic-retinopathy-224x224-gaussian-filtered'):
    level = root.replace('/kaggle/input/datasets/sovitrath/diabetic-retinopathy-224x224-gaussian-filtered', '').count(os.sep)
    if level < 3:
        indent = ' ' * 2 * level
        print(f'{indent}{os.path.basename(root)}/')
        # Show a couple sample files at each level
        for f in list(files)[:2]:
            print(f'{indent}  {f}')

## Cell 4: Data loading & EDA

In [ ]:
def load_aptos(root):
    # ── Find CSV ──────────────────────────────────────────────────────────
    for csv_name in ['train.csv', 'trainLabels.csv', 'train_labels.csv']:
        csv_path = f'{root}/{csv_name}'
        if os.path.exists(csv_path):
            df = pd.read_csv(csv_path)
            print(f'CSV loaded: {csv_path}')
            print(f'Columns   : {list(df.columns)}')
            break
    else:
        raise FileNotFoundError(
            f'No CSV found in {root}.\n'
            f'Available items: {os.listdir(root)}'
        )

    # ── Normalise column names ────────────────────────────────────────────
    df.columns = [c.lower().strip() for c in df.columns]
    if 'diagnosis' in df.columns:
        df = df.rename(columns={'diagnosis': 'label'})
    if 'id_code' not in df.columns and 'image' in df.columns:
        df = df.rename(columns={'image': 'id_code'})

    # ── Find image folder (checks nested path first) ──────────────────────
    for folder in [
        'gaussian_filtered_images/gaussian_filtered_images',
        'gaussian_filtered_images',
        'train_images',
        'resized_train_cropped',
        'train',
        'images'
    ]:
        candidate = f'{root}/{folder}'
        if os.path.isdir(candidate):
            img_folder = candidate
            print(f'Image folder: {img_folder}')
            break
    else:
        raise FileNotFoundError(
            f'No image folder found under {root}.\n'
            f'Available items: {os.listdir(root)}'
        )

    # ── Build full paths (searches class subfolders) ──────────────────────
    def make_path(img_id):
        for ext in ['.png', '.jpeg', '.jpg']:
            p = f'{img_folder}/{img_id}{ext}'
            if os.path.exists(p):
                return p
        for subfolder in os.listdir(img_folder):
            subfolder_path = f'{img_folder}/{subfolder}'
            if not os.path.isdir(subfolder_path):
                continue
            for ext in ['.png', '.jpeg', '.jpg']:
                p = f'{subfolder_path}/{img_id}{ext}'
                if os.path.exists(p):
                    return p
        return f'{img_folder}/{img_id}.png'

    df['path'] = df['id_code'].apply(make_path)
    df = df.rename(columns={'id_code': 'image_id'})

    # ── Verify a sample path exists ───────────────────────────────────────
    sample_path = df['path'].iloc[0]
    if not os.path.exists(sample_path):
        print(f'WARNING: Sample path does not exist: {sample_path}')
        print(f'First few files in folder: {os.listdir(img_folder)[:5]}')
    else:
        print(f'Path check OK: {sample_path}')

    return df[['image_id', 'label', 'path']]

df_aptos = load_aptos(CFG.APTOS_DIR)

# ── Reduce dataset to 50% for faster training ─────────────────────────────
from sklearn.model_selection import train_test_split

df_aptos, df_held_out = train_test_split(
    df_aptos,
    test_size=0.5,
    stratify=df_aptos['label'],
    random_state=SEED
)
df_aptos = df_aptos.reset_index(drop=True)
df_held_out = df_held_out.reset_index(drop=True)
print(f'\n✂️  Dataset reduced to 50%:')
print(f'   Training pool : {len(df_aptos)} images')
print(f'   Held-out test : {len(df_held_out)} images')

print('\nAPTOS 2019 — class distribution (after reduction):')
print(df_aptos['label'].value_counts().sort_index())

# ── Class weights for loss ────────────────────────────────────────────────
class_counts = df_aptos['label'].value_counts().sort_index().values
class_weights = 1.0 / class_counts
class_weights = class_weights / class_weights.sum() * CFG.N_CLASSES
CLASS_WEIGHTS = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)
print('\nClass weights:', CLASS_WEIGHTS.cpu().numpy().round(3))

# ── Visualise class distribution ──────────────────────────────────────────
grade_names = ['No DR (0)', 'Mild (1)', 'Moderate (2)', 'Severe (3)', 'Prolif. (4)']
colors = ['#27AE60','#F39C12','#E67E22','#E74C3C','#8E44AD']

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(grade_names, class_counts, color=colors, edgecolor='white', linewidth=1.2)
for bar, cnt in zip(bars, class_counts):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+15,
            f'{cnt}\n({cnt/len(df_aptos)*100:.1f}%)',
            ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_title('APTOS 2019 — DR Severity Grade Distribution (50% subset)',
             fontsize=13, fontweight='bold')
ax.set_ylabel('Number of images')
ax.set_ylim(0, max(class_counts)*1.25)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig(f'{CFG.OUT_DIR}/class_distribution.png', dpi=150)
plt.show()

# ── Sample images per grade ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for grade in range(5):
    sample = df_aptos[df_aptos['label']==grade].sample(1).iloc[0]
    img = Image.open(sample['path']).convert('RGB')
    img = img.resize((224,224))
    axes[grade].imshow(img)
    axes[grade].set_title(f'{grade_names[grade]}\n({sample["image_id"]})',
                          fontsize=10, color=colors[grade], fontweight='bold')
    axes[grade].axis('off')
plt.suptitle('Sample Fundus Images per DR Severity Grade', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{CFG.OUT_DIR}/sample_images.png', dpi=150)
plt.show()

## Cell 5: Preprocessing: CLAHE

In [ ]:
def apply_clahe(img_bgr, clip_limit=2.0, tile_grid=(8,8)):
    """Apply CLAHE to the green channel of a BGR image."""
    lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid)
    l_clahe = clahe.apply(l)
    lab_clahe = cv2.merge([l_clahe, a, b])
    return cv2.cvtColor(lab_clahe, cv2.COLOR_LAB2BGR)

def load_and_preprocess(path, size=CFG.IMG_SIZE):
    """Load image, apply CLAHE, resize."""
    img = cv2.imread(str(path))
    if img is None:
        img = np.zeros((size, size, 3), dtype=np.uint8)
    img = apply_clahe(img)
    img = cv2.resize(img, (size, size))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img   # uint8 H×W×3

# ── Show CLAHE effect ─────────────────────────────────────────────────────────
sample_path = df_aptos.sample(1).iloc[0]['path']
raw = cv2.cvtColor(cv2.imread(sample_path), cv2.COLOR_BGR2RGB)
raw_resized = cv2.resize(raw, (CFG.IMG_SIZE, CFG.IMG_SIZE))
clahe_img = load_and_preprocess(sample_path)

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].imshow(raw_resized); axes[0].set_title('Original', fontsize=12)
axes[1].imshow(clahe_img);   axes[1].set_title('After CLAHE', fontsize=12)
for ax in axes: ax.axis('off')
plt.suptitle('CLAHE Contrast Enhancement Effect', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{CFG.OUT_DIR}/clahe_effect.png', dpi=150)
plt.show()

## Cell 6: Albumentations augmentation pipelines

In [ ]:
train_aug = A.Compose([
    A.Resize(CFG.IMG_SIZE, CFG.IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Rotate(limit=30, border_mode=0, p=0.5),
    A.RandomBrightnessContrast(p=0.5),
    A.GaussianBlur(blur_limit=(0,3), p=0.2),
    A.Normalize(mean=CFG.MEAN, std=CFG.STD),
    ToTensorV2(),
])

val_aug = A.Compose([
    A.Resize(CFG.IMG_SIZE, CFG.IMG_SIZE),
    A.Normalize(mean=CFG.MEAN, std=CFG.STD),
    ToTensorV2(),
])

# ── Visualise augmentations ───────────────────────────────────────────────────
sample_img = load_and_preprocess(df_aptos.sample(1).iloc[0]['path'])
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes[0,0].imshow(sample_img); axes[0,0].set_title('Original (CLAHE)', fontweight='bold')
aug_vis = A.Compose([
    A.HorizontalFlip(p=1),
    A.VerticalFlip(p=1),
    A.Rotate(limit=30, p=1),
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=1),
    A.GaussianBlur(blur_limit=3, p=1),
    A.RandomScale(scale_limit=0.15, p=1),
])
aug_labels = ['H-Flip', 'V-Flip', 'Rotate', 'Brightness', 'Blur', 'Zoom']
for i, lbl in enumerate(aug_labels):
    ax = axes[(i+1)//4, (i+1)%4]
    aug_img = aug_vis(image=sample_img.copy())['image']
    ax.imshow(aug_img); ax.set_title(lbl, fontweight='bold')
for ax in axes.flat: ax.axis('off')
plt.suptitle('Data Augmentation Pipeline Visualisation', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{CFG.OUT_DIR}/augmentations.png', dpi=150)
plt.show()

## Cell 7: Dataset class

In [ ]:
class DRDataset(Dataset):
    def __init__(self, df, transform=None, preload=False):
        self.df        = df.reset_index(drop=True)
        self.transform = transform
        self.preload   = preload
        self.cache     = {}
        if preload:
            print("Preloading images into RAM...")
            for i, row in tqdm(self.df.iterrows(), total=len(self.df)):
                self.cache[i] = load_and_preprocess(row['path'])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        if self.preload and idx in self.cache:
            img = self.cache[idx]
        else:
            img = load_and_preprocess(self.df.loc[idx, 'path'])
        label = int(self.df.loc[idx, 'label'])
        if self.transform:
            img = self.transform(image=img)['image']
        return img, label

## Cell 8: Model architectures

In [ ]:
# ── 8a. EfficientNetB3-only baseline ─────────────────────────────────────────
class EfficientNetOnly(nn.Module):
    def __init__(self, n_classes=CFG.N_CLASSES, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(
            'efficientnet_b3', pretrained=pretrained,
            num_classes=0, global_pool='avg'
        )
        self.head = nn.Sequential(
            nn.Linear(1536, 512), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(512, 128),  nn.ReLU(),
            nn.Linear(128, n_classes)
        )
    def forward(self, x):
        feat = self.backbone(x)
        return self.head(feat)
    def get_features(self, x):
        return self.backbone(x)

# ── 8b. ViT-only baseline ─────────────────────────────────────────────────────
class ViTOnly(nn.Module):
    def __init__(self, n_classes=CFG.N_CLASSES, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(
            'vit_base_patch16_224', pretrained=pretrained,
            num_classes=0
        )
        self.head = nn.Sequential(
            nn.Linear(768, 512), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(512, 128), nn.ReLU(),
            nn.Linear(128, n_classes)
        )
    def forward(self, x):
        feat = self.backbone(x)
        return self.head(feat)
    def get_features(self, x):
        return self.backbone(x)

# ── 8c. CNN-ViT Hybrid (proposed) ────────────────────────────────────────────
class CNNViTHybrid(nn.Module):
    """
    Dual-stream hybrid:
      - EfficientNetB3  → 1,536-dim local feature vector
      - ViT-Base-Patch16-224 → 768-dim global context vector
      - Concat → 2,304-dim → FC-512 → Dropout(0.4) → FC-128 → FC-5
    """
    def __init__(self, n_classes=CFG.N_CLASSES, pretrained=True):
        super().__init__()
        # ── CNN stream ──
        self.cnn = timm.create_model(
            'efficientnet_b3', pretrained=pretrained,
            num_classes=0, global_pool='avg'
        )  # output: 1536
        # ── ViT stream ──
        self.vit = timm.create_model(
            'vit_base_patch16_224', pretrained=pretrained,
            num_classes=0
        )  # output: 768
        # ── Fusion head ──
        fused_dim = 1536 + 768  # 2304
        self.head = nn.Sequential(
            nn.Linear(fused_dim, 512), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(512, 128),       nn.ReLU(),
            nn.Linear(128, n_classes)
        )
        # Store feature maps for XAI
        self._cnn_features   = None
        self._cnn_gradients  = None
        self._vit_attn_maps  = None
        self._register_hooks()

    def _register_hooks(self):
        # Grad-CAM: hook on last conv block of EfficientNetB3
        target_layer = self.cnn.conv_head
        def forward_hook(module, inp, out):
            self._cnn_features = out
        def backward_hook(module, grad_in, grad_out):
            self._cnn_gradients = grad_out[0]
        target_layer.register_forward_hook(forward_hook)
        target_layer.register_backward_hook(backward_hook)

    def forward(self, x):
        cnn_feat = self.cnn(x)          # (B, 1536)
        vit_feat = self.vit(x)          # (B, 768)
        fused    = torch.cat([cnn_feat, vit_feat], dim=1)  # (B, 2304)
        return self.head(fused)

    def get_cnn_features(self, x):
        return self.cnn(x)

    def get_vit_features(self, x):
        return self.vit(x)

    def get_grad_cam(self, x, class_idx=None):
        """Generate Grad-CAM heatmap for a single image (B=1)."""
        self.zero_grad()
        logits = self.forward(x)
        if class_idx is None:
            class_idx = logits.argmax(dim=1).item()
        logits[0, class_idx].backward(retain_graph=True)
        # GAP over gradients
        weights = self._cnn_gradients.mean(dim=[2, 3], keepdim=True)  # (1,C,1,1)
        cam     = (weights * self._cnn_features).sum(dim=1, keepdim=True)
        cam     = F.relu(cam)
        cam     = F.interpolate(cam, size=(CFG.IMG_SIZE, CFG.IMG_SIZE),
                                mode='bilinear', align_corners=False)
        cam     = cam.squeeze().detach().cpu().numpy()
        cam     = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam, class_idx

    def get_vit_attention_rollout(self, x):
        """
        Compute ViT attention rollout for a single image (B=1).
        Returns a (14,14) attention map.
        """
        # Access all attention layers in the ViT encoder
        attn_maps = []
        hooks = []
        for blk in self.vit.blocks:
            def hook_fn(module, inp, out, storage=attn_maps):
                # timm ViT attn returns (B, heads, N, N)
                storage.append(out.detach().cpu())
            h = blk.attn.register_forward_hook(hook_fn)
            hooks.append(h)

        with torch.no_grad():
            _ = self.vit(x)
        for h in hooks:
            h.remove()

        # Rollout: product of attention matrices
        rollout = torch.eye(attn_maps[0].shape[-1])
        for attn in attn_maps:
            attn_avg = attn.mean(dim=1)          # avg over heads → (B,N,N)
            attn_avg = attn_avg[0]               # (N, N)
            attn_avg = attn_avg + torch.eye(attn_avg.shape[0])  # residual
            attn_avg = attn_avg / attn_avg.sum(dim=-1, keepdim=True)
            rollout  = torch.mm(attn_avg, rollout)

        # CLS token attends to all patches; reshape patch attention
        mask = rollout[0, 1:]  # (196,) — exclude CLS
        mask = mask.reshape(14, 14).numpy()
        mask = (mask - mask.min()) / (mask.max() - mask.min() + 1e-8)
        return mask

def build_model(name, pretrained=True):
    if name == 'efficientnet_only':
        return EfficientNetOnly(pretrained=pretrained).to(DEVICE)
    elif name == 'vit_only':
        return ViTOnly(pretrained=pretrained).to(DEVICE)
    elif name == 'hybrid':
        return CNNViTHybrid(pretrained=pretrained).to(DEVICE)
    else:
        raise ValueError(f"Unknown model: {name}")

# ── Quick architecture summary ─────────────────────────────────────────────────
model_tmp = build_model('hybrid')
total_params = sum(p.numel() for p in model_tmp.parameters())
trainable   = sum(p.numel() for p in model_tmp.parameters() if p.requires_grad)
print(f"Hybrid model — total params   : {total_params/1e6:.1f}M")
print(f"               trainable params: {trainable/1e6:.1f}M")
del model_tmp; gc.collect()

## Cell 9: Training utilities

In [ ]:
def get_loss_fn():
    return nn.CrossEntropyLoss(weight=CLASS_WEIGHTS)

def quadratic_weighted_kappa(y_true, y_pred):
    return cohen_kappa_score(y_true, y_pred, weights='quadratic')

class EarlyStopping:
    def __init__(self, patience=CFG.PATIENCE, mode='max', delta=1e-4):
        self.patience   = patience
        self.mode       = mode
        self.delta      = delta
        self.best       = -np.inf if mode=='max' else np.inf
        self.counter    = 0
        self.early_stop = False

    def __call__(self, score):
        improved = (score > self.best + self.delta) if self.mode=='max' \
                   else (score < self.best - self.delta)
        if improved:
            self.best    = score
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        return self.early_stop

def freeze_backbone(model):
    """Phase 1: freeze all backbone weights."""
    for name, param in model.named_parameters():
        if 'head' not in name:
            param.requires_grad = False

def unfreeze_all(model):
    """Phase 2: unfreeze everything."""
    for param in model.parameters():
        param.requires_grad = True

def get_optimizer(model, phase=1):
    if phase == 1:
        return AdamW(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=CFG.LR_HEAD, weight_decay=CFG.WEIGHT_DECAY
        )
    else:  # phase 2: differential LR
        backbone_params, head_params = [], []
        for name, param in model.named_parameters():
            if 'head' in name:
                head_params.append(param)
            else:
                backbone_params.append(param)
        return AdamW([
            {'params': backbone_params, 'lr': CFG.LR_BACKBONE},
            {'params': head_params,     'lr': CFG.LR_HEAD},
        ], weight_decay=CFG.WEIGHT_DECAY)

# ── One training epoch ────────────────────────────────────────────────────────
def train_epoch(model, loader, optimizer, criterion, scheduler=None):
    model.train()
    total_loss, all_preds, all_labels = 0.0, [], []
    for imgs, labels in tqdm(loader, leave=False, desc='  train'):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        logits = model(imgs)
        loss   = criterion(logits, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        if scheduler is not None:
            scheduler.step()
        total_loss += loss.item() * imgs.size(0)
        preds = logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())
    avg_loss = total_loss / len(loader.dataset)
    qwk      = quadratic_weighted_kappa(all_labels, all_preds)
    return avg_loss, qwk

# ── Validation epoch ──────────────────────────────────────────────────────────
def val_epoch(model, loader, criterion):
    model.eval()
    total_loss, all_preds, all_labels, all_probs = 0.0, [], [], []
    with torch.no_grad():
        for imgs, labels in tqdm(loader, leave=False, desc='  val  '):
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            logits = model(imgs)
            loss   = criterion(logits, labels)
            total_loss += loss.item() * imgs.size(0)
            probs = torch.softmax(logits, dim=1).cpu().numpy()
            preds = logits.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs)
    avg_loss = total_loss / len(loader.dataset)
    all_probs = np.array(all_probs)
    qwk  = quadratic_weighted_kappa(all_labels, all_preds)
    acc  = accuracy_score(all_labels, all_preds)
    f1   = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    # One-vs-rest AUC
    try:
        y_bin = label_binarize(all_labels, classes=list(range(CFG.N_CLASSES)))
        auc   = roc_auc_score(y_bin, all_probs, multi_class='ovr', average='macro')
    except Exception:
        auc = 0.0
    return avg_loss, qwk, acc, f1, auc, np.array(all_preds), np.array(all_labels)

In [ ]:
import os
files = sorted(os.listdir('/kaggle/working/outputs'))
for f in files:
    size = os.path.getsize(f'/kaggle/working/outputs/{f}')
    print(f'{f}  ({size/1e6:.1f} MB)')

## Cell 10: 5-Fold Cross-Validation training loop

In [ ]:
import json, shutil, glob, os

def save_results_json():
    results_serializable = {}
    for model_name, metrics in results.items():
        results_serializable[model_name] = {
            k: [float(x) for x in v] 
            for k, v in metrics.items()
        }
    with open(f'{CFG.OUT_DIR}/results.json', 'w') as f:
        json.dump(results_serializable, f, indent=2)
    print(f"📊 Results saved to results.json")

def permanent_backup(model_name, fold):
    src = f'{CFG.OUT_DIR}/{model_name}_fold{fold}_best.pth'
    dst = f'/kaggle/working/{model_name}_fold{fold}_best.pth'
    if os.path.exists(src):
        shutil.copy2(src, dst)
        print(f"✅ Backed up: {model_name} fold {fold}")

def cleanup_old_checkpoints():
    """Delete per-epoch checkpoint files to free disk space"""
    removed = 0
    for f in glob.glob(f'{CFG.OUT_DIR}/*_ckpt_epoch*.pth'):
        os.remove(f)
        removed += 1
    if removed:
        print(f"🗑️ Removed {removed} old epoch checkpoints to free disk space")

def is_fold_done(model_name, fold):
    """Check if a fold is already completed and backed up"""
    backup_path = f'/kaggle/working/{model_name}_fold{fold}_best.pth'
    return os.path.exists(backup_path)

def load_results_json():
    """Load previously saved results if available"""
    path = f'{CFG.OUT_DIR}/results.json'
    if os.path.exists(path):
        with open(path, 'r') as f:
            saved = json.load(f)
        print(f"📂 Loaded previous results from results.json")
        return saved
    return None

# ── Clean up old epoch checkpoints first ─────────────────────────────────────
cleanup_old_checkpoints()

# ── Load previous results if available ───────────────────────────────────────
skf = StratifiedKFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=SEED)

results = {m: {'fold_qwk':[], 'fold_acc':[], 'fold_f1':[], 'fold_auc':[]}
           for m in CFG.MODELS}

saved_results = load_results_json()
if saved_results:
    for model_name in CFG.MODELS:
        for metric in ['fold_qwk', 'fold_acc', 'fold_f1', 'fold_auc']:
            if model_name in saved_results and metric in saved_results[model_name]:
                results[model_name][metric] = saved_results[model_name][metric]
    print("✅ Previous results restored:")
    for model_name in CFG.MODELS:
        n = len(results[model_name]['fold_qwk'])
        if n > 0:
            print(f"   {model_name}: {n} folds done, "
                  f"mean QWK={sum(results[model_name]['fold_qwk'])/n:.4f}")

# ── Training loop ─────────────────────────────────────────────────────────────
for model_name in CFG.MODELS:
    print(f"\n{'='*60}")
    print(f"  Training: {model_name.upper()}")
    print(f"{'='*60}")

    for fold, (train_idx, val_idx) in enumerate(
            skf.split(df_aptos, df_aptos['label'])):

        # ── Skip already completed folds ──────────────────────────
        if is_fold_done(model_name, fold+1):
            print(f"\n  ── Fold {fold+1}/{CFG.N_FOLDS} — ⏭️ SKIPPED (already done) ──")
            continue

        print(f"\n  ── Fold {fold+1}/{CFG.N_FOLDS} ──")
        seed_everything()

        df_train = df_aptos.iloc[train_idx]
        df_val   = df_aptos.iloc[val_idx]

        train_ds = DRDataset(df_train, transform=train_aug)
        val_ds   = DRDataset(df_val,   transform=val_aug)

        train_dl = DataLoader(train_ds, batch_size=CFG.BATCH_SIZE,
                              shuffle=True,  num_workers=CFG.NUM_WORKERS,
                              pin_memory=True)
        val_dl   = DataLoader(val_ds,   batch_size=CFG.BATCH_SIZE,
                              shuffle=False, num_workers=CFG.NUM_WORKERS,
                              pin_memory=True)

        model     = build_model(model_name)
        criterion = get_loss_fn()
        es        = EarlyStopping(patience=CFG.PATIENCE, mode='max')

        best_qwk      = -1.0
        best_weights  = None
        history       = {'train_loss':[], 'val_loss':[], 'train_qwk':[], 'val_qwk':[]}

        # ── Phase 1: frozen backbone (epochs 1-10) ────────────────
        freeze_backbone(model)
        optimizer  = get_optimizer(model, phase=1)
        scheduler  = CosineAnnealingWarmRestarts(optimizer, T_0=CFG.T0)

        for epoch in range(1, CFG.EPOCHS + 1):
            if epoch == 11:
                unfreeze_all(model)
                optimizer = get_optimizer(model, phase=2)
                scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=CFG.T0)
                print(f"    [Epoch {epoch}] Phase 2: unfrozen backbone, differential LR")

            t_loss, t_qwk = train_epoch(model, train_dl, optimizer, criterion, scheduler)
            v_loss, v_qwk, v_acc, v_f1, v_auc, _, _ = val_epoch(model, val_dl, criterion)

            history['train_loss'].append(t_loss)
            history['val_loss'].append(v_loss)
            history['train_qwk'].append(t_qwk)
            history['val_qwk'].append(v_qwk)

            if v_qwk > best_qwk:
                best_qwk     = v_qwk
                best_weights = {k: v.clone() for k, v in model.state_dict().items()}
                # ── Save only ONE checkpoint file per fold (overwrites) ──
                torch.save(best_weights,
                           f'{CFG.OUT_DIR}/{model_name}_fold{fold+1}_best.pth')
                torch.save({
                    'model_name' : model_name,
                    'fold'       : fold + 1,
                    'epoch'      : epoch,
                    'qwk'        : v_qwk,
                    'state_dict' : best_weights,
                    'history'    : history,
                }, f'{CFG.OUT_DIR}/{model_name}_fold{fold+1}_ckpt_best.pth')  # ← overwrites same file
                print(f"    💾 Saved checkpoint (QWK={v_qwk:.4f})")

            print(f"    Epoch {epoch:02d} | "
                  f"T-Loss {t_loss:.4f} T-QWK {t_qwk:.4f} | "
                  f"V-Loss {v_loss:.4f} V-QWK {v_qwk:.4f} "
                  f"V-Acc {v_acc:.4f} V-AUC {v_auc:.4f}"
                  + (' ★' if v_qwk == best_qwk else ''))

            if es(v_qwk):
                print(f"    Early stopping at epoch {epoch}")
                break

        # ── Load best weights, final evaluation ──────────────────
        model.load_state_dict(best_weights)
        _, f_qwk, f_acc, f_f1, f_auc, f_preds, f_labels = \
            val_epoch(model, val_dl, criterion)

        results[model_name]['fold_qwk'].append(f_qwk)
        results[model_name]['fold_acc'].append(f_acc)
        results[model_name]['fold_f1'].append(f_f1)
        results[model_name]['fold_auc'].append(f_auc)

        print(f"\n  Fold {fold+1} best → QWK={f_qwk:.4f}  Acc={f_acc:.4f}  "
              f"F1={f_f1:.4f}  AUC={f_auc:.4f}")

        # ── Confusion matrix per fold ─────────────────────────────
        cm = confusion_matrix(f_labels, f_preds)
        fig, ax = plt.subplots(figsize=(7,6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=grade_names, yticklabels=grade_names, ax=ax)
        ax.set_title(f'{model_name} — Fold {fold+1} Confusion Matrix\n'
                     f'QWK={f_qwk:.4f}', fontsize=12, fontweight='bold')
        ax.set_xlabel('Predicted'); ax.set_ylabel('True')
        plt.tight_layout()
        plt.savefig(f'{CFG.OUT_DIR}/{model_name}_fold{fold+1}_cm.png', dpi=120)
        plt.close()

        # ── Training curves ───────────────────────────────────────
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        ep = range(1, len(history['train_loss'])+1)
        axes[0].plot(ep, history['train_loss'], label='Train')
        axes[0].plot(ep, history['val_loss'],   label='Val')
        axes[0].set_title('Loss'); axes[0].legend()
        axes[1].plot(ep, history['train_qwk'],  label='Train')
        axes[1].plot(ep, history['val_qwk'],    label='Val')
        axes[1].set_title('QWK');  axes[1].legend()
        plt.suptitle(f'{model_name} — Fold {fold+1} Training Curves', fontweight='bold')
        plt.tight_layout()
        plt.savefig(f'{CFG.OUT_DIR}/{model_name}_fold{fold+1}_curves.png', dpi=120)
        plt.close()

        # ── Permanent backup after each fold ─────────────────────
        permanent_backup(model_name, fold+1)
        save_results_json()

        # ── Clean up checkpoint to free disk space ────────────────
        ckpt = f'{CFG.OUT_DIR}/{model_name}_fold{fold+1}_ckpt_best.pth'
        if os.path.exists(ckpt):
            os.remove(ckpt)
            print(f"🗑️ Cleaned up checkpoint for {model_name} fold {fold+1}")

        del model; gc.collect()
        torch.cuda.empty_cache()

In [ ]:
import inspect
print(inspect.getsource(val_aug.__class__))

In [ ]:
print(train_aug)
print(val_aug)

## Cell 11: Cross-fold results summary

In [ ]:
print("\n" + "="*65)
print("  CROSS-VALIDATION RESULTS SUMMARY")
print("="*65)

summary_rows = []
for model_name in CFG.MODELS:
    r = results[model_name]
    row = {
        'Model'    : model_name,
        'QWK Mean' : np.mean(r['fold_qwk']),
        'QWK Std'  : np.std(r['fold_qwk']),
        'Acc Mean' : np.mean(r['fold_acc']),
        'F1 Mean'  : np.mean(r['fold_f1']),
        'AUC Mean' : np.mean(r['fold_auc']),
    }
    summary_rows.append(row)
    print(f"\n  {model_name.upper()}")
    for fold_i, qwk in enumerate(r['fold_qwk']):
        print(f"    Fold {fold_i+1}: QWK={qwk:.4f}")
    print(f"    ── Mean QWK : {row['QWK Mean']:.4f} ± {row['QWK Std']:.4f}")
    print(f"    ── Mean Acc : {row['Acc Mean']:.4f}")
    print(f"    ── Mean F1  : {row['F1 Mean']:.4f}")
    print(f"    ── Mean AUC : {row['AUC Mean']:.4f}")

df_summary = pd.DataFrame(summary_rows)
df_summary.to_csv(f'{CFG.OUT_DIR}/cv_results_summary.csv', index=False)
print("\nSaved: cv_results_summary.csv")

# ── Grouped bar chart — QWK per fold ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
x     = np.arange(CFG.N_FOLDS)
width = 0.25
bar_cols = ['#2980B9','#27AE60','#E74C3C']
labels   = ['EfficientNetB3-Only','ViT-Only','CNN-ViT Hybrid (Proposed)']
for i, (model_name, col, lbl) in enumerate(zip(CFG.MODELS, bar_cols, labels)):
    vals = results[model_name]['fold_qwk']
    bars = ax.bar(x + i*width, vals, width, label=lbl, color=col,
                  alpha=0.85, edgecolor='white')
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                f'{v:.3f}', ha='center', va='bottom', fontsize=8)
ax.set_xlabel('Fold'); ax.set_ylabel('Quadratic Weighted Kappa')
ax.set_title('QWK per Fold — Ablation Comparison', fontsize=13, fontweight='bold')
ax.set_xticks(x + width)
ax.set_xticklabels([f'Fold {i+1}' for i in range(CFG.N_FOLDS)])
ax.legend(); ax.set_ylim(0, 1.05)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig(f'{CFG.OUT_DIR}/qwk_per_fold.png', dpi=150)
plt.show()

## Cell 12: Paired t-test (Hybrid vs baselines)

In [ ]:
print("\n" + "="*55)
print("  PAIRED t-TEST  (α = 0.05, two-sided)")
print("="*55)

hybrid_qwks = results['hybrid']['fold_qwk']

for baseline in ['efficientnet_only', 'vit_only']:
    base_qwks = results[baseline]['fold_qwk']
    diffs = [h - b for h, b in zip(hybrid_qwks, base_qwks)]
    t_stat, p_val = stats.ttest_rel(hybrid_qwks, base_qwks)
    mean_diff = np.mean(diffs)
    ci_low    = mean_diff - stats.t.ppf(0.975, df=CFG.N_FOLDS-1) * np.std(diffs,ddof=1)/np.sqrt(CFG.N_FOLDS)
    ci_high   = mean_diff + stats.t.ppf(0.975, df=CFG.N_FOLDS-1) * np.std(diffs,ddof=1)/np.sqrt(CFG.N_FOLDS)
    sig = "✓ SIGNIFICANT" if p_val < 0.05 else "✗ NOT significant"
    print(f"\n  Hybrid vs {baseline}")
    print(f"    Mean ΔQWK : {mean_diff:+.4f}  (95% CI: [{ci_low:+.4f}, {ci_high:+.4f}])")
    print(f"    t-statistic: {t_stat:.4f}")
    print(f"    p-value    : {p_val:.4f}  →  {sig}")

# ── Save t-test results ───────────────────────────────────────────────────────
ttest_rows = []
for baseline in ['efficientnet_only', 'vit_only']:
    base_qwks = results[baseline]['fold_qwk']
    t_stat, p_val = stats.ttest_rel(hybrid_qwks, base_qwks)
    ttest_rows.append({
        'Comparison': f'Hybrid vs {baseline}',
        'Mean_dQWK' : np.mean([h-b for h,b in zip(hybrid_qwks,base_qwks)]),
        't_stat'    : t_stat,
        'p_value'   : p_val,
        'Significant': p_val < 0.05
    })
pd.DataFrame(ttest_rows).to_csv(f'{CFG.OUT_DIR}/ttest_results.csv', index=False)

## Cell 13: Per-class F1 breakdown (best fold of hybrid)

In [ ]:
best_fold = int(np.argmax(results['hybrid']['fold_qwk']))
print(f"\nBest hybrid fold: Fold {best_fold+1} "
      f"(QWK = {results['hybrid']['fold_qwk'][best_fold]:.4f})")

# Load best model weights
model_eval = build_model('hybrid')
model_eval.load_state_dict(
    torch.load(f'{CFG.OUT_DIR}/hybrid_fold{best_fold+1}_best.pth',
               map_location=DEVICE)
)
model_eval.eval()

# Get val set for that fold
fold_splits = list(skf.split(df_aptos, df_aptos['label']))
_, val_idx_best = fold_splits[best_fold]
df_val_best = df_aptos.iloc[val_idx_best]
val_dl_best = DataLoader(
    DRDataset(df_val_best, transform=val_aug),
    batch_size=CFG.BATCH_SIZE, shuffle=False, num_workers=CFG.NUM_WORKERS
)

all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for imgs, labels in tqdm(val_dl_best, desc='Evaluating best fold'):
        logits = model_eval(imgs.to(DEVICE))
        probs  = torch.softmax(logits, dim=1).cpu().numpy()
        preds  = logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds); all_labels.extend(labels.numpy())
        all_probs.extend(probs)

all_probs  = np.array(all_probs)
all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

print("\nClassification Report (Best Hybrid Fold):")
print(classification_report(all_labels, all_preds,
                             target_names=grade_names, zero_division=0))

# ── Per-class F1 bar chart ────────────────────────────────────────────────────
f1_per_class = f1_score(all_labels, all_preds, average=None, zero_division=0)
fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(grade_names, f1_per_class, color=colors, edgecolor='white', linewidth=1.2)
for bar, val in zip(bars, f1_per_class):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
            f'{val:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.axhline(0.8, color='gray', linestyle='--', linewidth=1.2, label='0.80 clinical threshold')
ax.set_ylim(0, 1.1); ax.set_ylabel('F1-Score')
ax.set_title('Per-Class F1-Score — CNN-ViT Hybrid (Best Fold)', fontsize=13, fontweight='bold')
ax.legend(); ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig(f'{CFG.OUT_DIR}/per_class_f1.png', dpi=150)
plt.show()

# ── Multi-class ROC curves ────────────────────────────────────────────────────
from sklearn.metrics import roc_curve, auc as sk_auc
y_bin = label_binarize(all_labels, classes=list(range(CFG.N_CLASSES)))
fig, ax = plt.subplots(figsize=(8, 6))
for i, (gname, col) in enumerate(zip(grade_names, colors)):
    fpr, tpr, _ = roc_curve(y_bin[:, i], all_probs[:, i])
    roc_auc     = sk_auc(fpr, tpr)
    ax.plot(fpr, tpr, color=col, lw=2, label=f'{gname}  (AUC={roc_auc:.3f})')
ax.plot([0,1],[0,1],'k--',lw=1.2)
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('One-vs-Rest ROC Curves — CNN-ViT Hybrid', fontsize=13, fontweight='bold')
ax.legend(loc='lower right', fontsize=9)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig(f'{CFG.OUT_DIR}/roc_curves.png', dpi=150)
plt.show()

## Cell 14: XAI: Grad-CAM generation & visualisation

In [ ]:
def visualise_grad_cam(model, img_tensor, img_original, label, path,
                       class_idx=None, save_path=None):
    """
    img_tensor  : (1,3,224,224) on DEVICE
    img_original: (224,224,3) uint8 numpy
    """
    model.eval()
    cam, pred_class = model.get_grad_cam(img_tensor, class_idx)
    # Threshold at 70th percentile
    thresh = np.percentile(cam, CFG.SALIENCY_THRESH_PERCENTILE)
    cam_binary = (cam >= thresh).astype(np.uint8)

    # Overlay
    heatmap = cv2.applyColorMap(
        (cam * 255).astype(np.uint8), cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    overlay = (0.55 * img_original + 0.45 * heatmap).astype(np.uint8)

    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    axes[0].imshow(img_original); axes[0].set_title('Original (CLAHE)')
    axes[1].imshow(cam, cmap='hot'); axes[1].set_title(f'Grad-CAM\n(pred: {grade_names[pred_class]})')
    axes[2].imshow(cam_binary, cmap='gray'); axes[2].set_title(f'Binary Mask\n(70th pct threshold)')
    axes[3].imshow(overlay); axes[3].set_title(f'Overlay\n(true: {grade_names[label]})')
    for ax in axes: ax.axis('off')
    plt.suptitle(f'Grad-CAM — {Path(path).stem}', fontweight='bold')
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=130, bbox_inches='tight')
    plt.show()
    return cam, cam_binary

# ── Generate Grad-CAM for one sample per grade ────────────────────────────────
print("\nGenerating Grad-CAM visualisations...")
for grade in range(CFG.N_CLASSES):
    row = df_aptos[df_aptos['label'] == grade].sample(1, random_state=SEED).iloc[0]
    img_np     = load_and_preprocess(row['path'])
    img_tensor = val_aug(image=img_np)['image'].unsqueeze(0).to(DEVICE)
    visualise_grad_cam(
        model_eval, img_tensor, img_np,
        label=grade, path=row['path'],
        save_path=f'{CFG.OUT_DIR}/grad_cam_grade{grade}.png'
    )

## Cell 15: XAI: ViT Attention Rollout

In [ ]:
def visualise_attention_rollout(model, img_tensor, img_original, label,
                                 save_path=None):
    model.eval()
    attn_map = model.get_vit_attention_rollout(img_tensor)  # (14,14)
    # Upsample to 224×224
    attn_up  = cv2.resize(attn_map, (CFG.IMG_SIZE, CFG.IMG_SIZE),
                           interpolation=cv2.INTER_LINEAR)
    thresh   = np.percentile(attn_up, CFG.SALIENCY_THRESH_PERCENTILE)
    attn_bin = (attn_up >= thresh).astype(np.uint8)

    heatmap  = cv2.applyColorMap((attn_up*255).astype(np.uint8), cv2.COLORMAP_VIRIDIS)
    heatmap  = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    overlay  = (0.55 * img_original + 0.45 * heatmap).astype(np.uint8)

    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    axes[0].imshow(img_original);     axes[0].set_title('Original (CLAHE)')
    axes[1].imshow(attn_map, cmap='viridis');  axes[1].set_title('Attn Rollout (14×14)')
    axes[2].imshow(attn_bin, cmap='gray');     axes[2].set_title('Binary Mask (70th pct)')
    axes[3].imshow(overlay);           axes[3].set_title(f'Overlay (grade {label})')
    for ax in axes: ax.axis('off')
    plt.suptitle('ViT Attention Rollout Visualisation', fontweight='bold')
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=130, bbox_inches='tight')
    plt.show()
    return attn_up, attn_bin

print("\nGenerating ViT Attention Rollout visualisations...")
for grade in range(CFG.N_CLASSES):
    row = df_aptos[df_aptos['label'] == grade].sample(1, random_state=SEED+1).iloc[0]
    img_np     = load_and_preprocess(row['path'])
    img_tensor = val_aug(image=img_np)['image'].unsqueeze(0).to(DEVICE)
    visualise_attention_rollout(
        model_eval, img_tensor, img_np, label=grade,
        save_path=f'{CFG.OUT_DIR}/vit_attention_grade{grade}.png'
    )

## Cell 16: XAI Quantitative Benchmarking: IoU & mAP vs IDRiD

In [ ]:
"""
IDRiD dataset structure on Kaggle:
  /kaggle/input/idrid-diabetic-retinopathy-segmentation/
    ├── A. Segmentation/
    │   ├── 1. Original Images/
    │   │   ├── a. Training Set/   (IDRiD_01.jpg … IDRiD_54.jpg)
    │   │   └── b. Testing Set/    (IDRiD_55.jpg … IDRiD_81.jpg)
    │   └── 2. All Segmentation Groundtruths/
    │       ├── a. Training Set/
    │       │   ├── 1. Microaneurysms/    (IDRiD_01_MA.tif …)
    │       │   ├── 2. Haemorrhages/      (IDRiD_01_HE.tif …)
    │       │   ├── 3. Hard Exudates/     (IDRiD_01_EX.tif …)
    │       │   └── 4. Soft Exudates/     (IDRiD_01_SE.tif …)
    │       └── b. Testing Set/  (same structure)
"""

IDRID_IMG_DIR  = Path(CFG.IDRID_DIR) / 'A. Segmentation/1. Original Images/b. Testing Set'
IDRID_MASK_DIR = Path(CFG.IDRID_DIR) / 'A. Segmentation/2. All Segmentation Groundtruths/b. Testing Set'
LESION_DIRS    = {
    'MA': '1. Microaneurysms',
    'HE': '2. Haemorrhages',
    'EX': '3. Hard Exudates',
    'SE': '4. Soft Exudates',
}

def compute_iou(pred_mask, gt_mask):
    """Compute IoU between two binary masks."""
    intersection = np.logical_and(pred_mask, gt_mask).sum()
    union        = np.logical_or(pred_mask, gt_mask).sum()
    if union == 0:
        return np.nan
    return float(intersection) / float(union)

def compute_ap(pred_map, gt_mask, thresholds=None):
    """Compute Average Precision by sweeping threshold."""
    if thresholds is None:
        thresholds = np.linspace(0, 1, 21)
    precisions, recalls = [], []
    for thr in thresholds:
        pred_bin = (pred_map >= thr).astype(np.uint8)
        tp = np.logical_and(pred_bin, gt_mask).sum()
        fp = np.logical_and(pred_bin, ~gt_mask.astype(bool)).sum()
        fn = np.logical_and(~pred_bin.astype(bool), gt_mask).sum()
        prec = tp / (tp + fp + 1e-8)
        rec  = tp / (tp + fn + 1e-8)
        precisions.append(prec)
        recalls.append(rec)
    # Area under P-R curve (trapezoidal)
    recalls    = np.array(recalls)
    precisions = np.array(precisions)
    order = np.argsort(recalls)
    ap = np.trapz(precisions[order], recalls[order])
    return float(ap)

def load_idrid_mask(mask_path, target_size=(CFG.IMG_SIZE, CFG.IMG_SIZE)):
    """Load IDRiD binary lesion mask, resize to target size."""
    if not Path(mask_path).exists():
        return None
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        return None
    mask = cv2.resize(mask, target_size, interpolation=cv2.INTER_NEAREST)
    return (mask > 127).astype(np.uint8)

def xai_benchmark_idrid(model, img_dir, mask_dir, lesion_dirs,
                         max_images=27):
    """
    Run quantitative XAI benchmarking:
      - Generate Grad-CAM and ViT attention maps for each IDRiD test image
      - Compare binary saliency masks against each lesion annotation type
      - Report per-lesion IoU and AP, then aggregate mAP

    Returns a DataFrame of results.
    """
    img_paths = sorted(Path(img_dir).glob('*.jpg'))[:max_images]
    rows = []

    print(f"\nBenchmarking on {len(img_paths)} IDRiD test images...")
    for img_path in tqdm(img_paths):
        stem = img_path.stem  # e.g. IDRiD_55

        # Load and preprocess image
        img_np     = load_and_preprocess(str(img_path))
        img_tensor = val_aug(image=img_np)['image'].unsqueeze(0).to(DEVICE)

        # Grad-CAM
        try:
            grad_cam_map, pred_cls = model.get_grad_cam(img_tensor)
        except Exception as e:
            print(f"  Grad-CAM failed for {stem}: {e}")
            continue

        # ViT Attention Rollout
        try:
            vit_attn_map = model.get_vit_attention_rollout(img_tensor)
            vit_attn_map = cv2.resize(vit_attn_map,
                                       (CFG.IMG_SIZE, CFG.IMG_SIZE),
                                       interpolation=cv2.INTER_LINEAR)
        except Exception as e:
            print(f"  ViT rollout failed for {stem}: {e}")
            vit_attn_map = np.zeros((CFG.IMG_SIZE, CFG.IMG_SIZE))

        # Binary saliency masks (70th percentile threshold)
        thr_gc = np.percentile(grad_cam_map, CFG.SALIENCY_THRESH_PERCENTILE)
        gc_bin = (grad_cam_map >= thr_gc).astype(np.uint8)

        thr_vit = np.percentile(vit_attn_map, CFG.SALIENCY_THRESH_PERCENTILE)
        vit_bin = (vit_attn_map >= thr_vit).astype(np.uint8)

        for lesion_key, lesion_subdir in lesion_dirs.items():
            # Find the mask file (naming: IDRiD_55_MA.tif etc.)
            mask_path = Path(mask_dir) / lesion_subdir / f'{stem}_{lesion_key}.tif'
            gt_mask   = load_idrid_mask(mask_path)
            if gt_mask is None or gt_mask.sum() == 0:
                continue  # skip if no annotation exists

            row = {'image': stem, 'lesion': lesion_key, 'pred_grade': pred_cls}

            # Grad-CAM metrics
            row['gc_iou'] = compute_iou(gc_bin,  gt_mask)
            row['gc_ap']  = compute_ap(grad_cam_map, gt_mask)

            # ViT metrics
            row['vit_iou'] = compute_iou(vit_bin,    gt_mask)
            row['vit_ap']  = compute_ap(vit_attn_map, gt_mask)

            rows.append(row)

    df_xai = pd.DataFrame(rows)
    return df_xai

# ── Run benchmarking ──────────────────────────────────────────────────────────
if IDRID_IMG_DIR.exists():
    df_xai = xai_benchmark_idrid(
        model_eval, IDRID_IMG_DIR, IDRID_MASK_DIR, LESION_DIRS
    )
    df_xai.to_csv(f'{CFG.OUT_DIR}/xai_benchmark_results.csv', index=False)

    # ── Aggregate results ─────────────────────────────────────────────────────
    print("\n" + "="*55)
    print("  XAI QUANTITATIVE BENCHMARKING — RESULTS")
    print("="*55)

    agg = df_xai.groupby('lesion').agg(
        GradCAM_IoU=('gc_iou',  'mean'),
        GradCAM_AP =('gc_ap',   'mean'),
        ViT_IoU    =('vit_iou', 'mean'),
        ViT_AP     =('vit_ap',  'mean'),
    ).round(4)
    print(agg.to_string())

    # Overall mAP
    gc_map  = df_xai['gc_ap'].mean()
    vit_map = df_xai['vit_ap'].mean()
    gc_miou  = df_xai['gc_iou'].mean()
    vit_miou = df_xai['vit_iou'].mean()
    print(f"\n  Grad-CAM :  mIoU = {gc_miou:.4f}   mAP = {gc_map:.4f}")
    print(f"  ViT Attn :  mIoU = {vit_miou:.4f}   mAP = {vit_map:.4f}")

    # ── XAI bar chart ─────────────────────────────────────────────────────────
    lesions    = agg.index.tolist()
    x          = np.arange(len(lesions))
    fig, axes  = plt.subplots(1, 2, figsize=(13, 5))

    for ax, metric, gc_col, vit_col, ylabel in [
        (axes[0], 'IoU',  'GradCAM_IoU', 'ViT_IoU', 'Mean IoU'),
        (axes[1], 'AP',   'GradCAM_AP',  'ViT_AP',  'Average Precision'),
    ]:
        ax.bar(x - 0.2, agg[gc_col],  0.35, label='Grad-CAM',    color='#2980B9', alpha=0.85)
        ax.bar(x + 0.2, agg[vit_col], 0.35, label='ViT Attention',color='#27AE60', alpha=0.85)
        ax.set_xticks(x); ax.set_xticklabels(lesions)
        ax.set_title(f'XAI {metric} vs IDRiD Lesion Masks', fontweight='bold')
        ax.set_ylabel(ylabel); ax.legend()
        ax.axhline(0.5, color='gray', linestyle='--', linewidth=1, alpha=0.7)
        ax.spines[['top','right']].set_visible(False)

    plt.suptitle('Quantitative XAI Benchmarking — Grad-CAM vs ViT Attention',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{CFG.OUT_DIR}/xai_benchmark_chart.png', dpi=150)
    plt.show()

    # ── Statistical comparison: Grad-CAM IoU vs ViT IoU ─────────────────────
    gc_ious  = df_xai['gc_iou'].dropna().values
    vit_ious = df_xai['vit_iou'].dropna().values
    common   = min(len(gc_ious), len(vit_ious))
    t_stat, p_val = stats.ttest_rel(gc_ious[:common], vit_ious[:common])
    print(f"\n  XAI IoU comparison (Grad-CAM vs ViT):")
    print(f"    t={t_stat:.4f}, p={p_val:.4f}  "
          + ("→ Significantly different" if p_val<0.05 else "→ Not significantly different"))

else:
    print(f"\nIDRiD dataset not found at {IDRID_IMG_DIR}")
    print("→ Add the IDRiD dataset to your Kaggle notebook and re-run this cell.")

## Cell 17: Final results table for paper (Section 5)

In [ ]:
print("\n" + "="*65)
print("  FINAL RESULTS — READY FOR PAPER TABLE")
print("="*65)

metrics_display = []
for model_name, label in zip(
        CFG.MODELS,
        ['EfficientNetB3-Only (Baseline A)',
         'ViT-Base-Patch16-224-Only (Baseline B)',
         'CNN-ViT Hybrid (Proposed)']):
    r = results[model_name]
    metrics_display.append({
        'Model'           : label,
        'QWK Mean ± Std'  : f"{np.mean(r['fold_qwk']):.4f} ± {np.std(r['fold_qwk']):.4f}",
        'Accuracy'        : f"{np.mean(r['fold_acc']):.4f}",
        'Macro F1'        : f"{np.mean(r['fold_f1']):.4f}",
        'Macro AUC'       : f"{np.mean(r['fold_auc']):.4f}",
    })

df_paper = pd.DataFrame(metrics_display)
print(df_paper.to_string(index=False))
df_paper.to_csv(f'{CFG.OUT_DIR}/paper_results_table.csv', index=False)

# ── Radar chart — model comparison ────────────────────────────────────────────
import matplotlib
categories = ['QWK', 'Accuracy', 'Macro F1', 'Macro AUC']
N = len(categories)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(7,7), subplot_kw=dict(polar=True))
model_colors = ['#2980B9','#27AE60','#E74C3C']
model_labels = ['EfficientNetB3-Only','ViT-Only','CNN-ViT Hybrid']

for model_name, col, lbl in zip(CFG.MODELS, model_colors, model_labels):
    r = results[model_name]
    vals = [np.mean(r['fold_qwk']), np.mean(r['fold_acc']),
            np.mean(r['fold_f1']),  np.mean(r['fold_auc'])]
    vals += vals[:1]
    ax.plot(angles, vals, 'o-', linewidth=2, color=col, label=lbl)
    ax.fill(angles, vals, alpha=0.1, color=col)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=12)
ax.set_ylim(0, 1)
ax.set_title('Model Comparison — All Metrics', fontsize=13,
             fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15))
plt.tight_layout()
plt.savefig(f'{CFG.OUT_DIR}/radar_chart.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nAll outputs saved to: {CFG.OUT_DIR}")
print("Files generated:")
for f in sorted(Path(CFG.OUT_DIR).glob('*')):
    print(f"  {f.name}")